# 07 · Análises — respostas às perguntas de negócio

Consultas sobre a camada Gold. Uma consulta por célula.

| Pergunta | Janela | Medida principal |
| --- | --- | --- |
| P1 — Quais cadeias produtivas mais cresceram em arrecadação de ICMS? | 2020–2025 | Crescimento relativo ao ICMS total do RS |
| P2 — Quais cadeias e COREDEs concentram as desonerações de ICMS, e qual a razão desoneração/arrecadação? | 2020–2025 | Participação e razão sobre o ICMS |
| P3 — Qual a participação de MPE nas empresas ativas por cadeia e COREDE, e como evoluiu a abertura? | 2020–2025 | Participação do Simples Nacional no estoque |
| P4 — Cadeias mais desoneradas tiveram maior saldo de empresas? | 2020–2025 | Correlação de Spearman entre cadeias |
| P5 — Municípios com maior crescimento do PIB per capita têm maior abertura de pequenos negócios? | 2018–2021 | Correlação de Spearman entre municípios |

Regras comuns:

- Valores em reais correntes. Conclusões em participação, ordenação e razão, nunca em crescimento real.
- Cadeias: as 14 prioritárias. A soma entre cadeias excede o total em cerca de 3% (ponte N:N); nenhuma consulta soma cadeias.
- Desoneração setorial: ICMS, nível detalhado, finalidade econômica e `integra_total_estadual`. O não estorno do crédito fica fora.
- MPE: categoria Simples Nacional. MEI fica fora (série desde set/2024); produtor rural nunca é somado às empresas.
- Resultados indicam associação, não causalidade.

In [0]:
%sql
USE CATALOG mvp_pipeline_vf;

## P1 — Crescimento do ICMS por cadeia produtiva

### 1.1 Série anual

Gráfico: linhas, eixo X `ano`, eixo Y `icms_milhoes`, série por `cadeia`.

In [0]:
%sql
SELECT t.ano,
       p.cadeia,
       round(sum(f.valor_icms) / 1e6, 1) AS icms_milhoes
FROM gold.fato_icms_cnae f
JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
WHERE t.ano BETWEEN 2020 AND 2025
GROUP BY t.ano, p.cadeia
ORDER BY p.cadeia, t.ano;

### 1.2 Ordenação das cadeias

`crescimento_relativo_perc` compara a variação da cadeia com a do ICMS total do RS no mesmo período. Como as duas
séries carregam a mesma inflação, a medida é neutra em relação a preços: valor positivo indica cadeia que cresceu
acima do Estado. A ordenação por essa medida coincide com a ordenação pela variação nominal.

`var_participacao_pp` mede o ganho de peso no ICMS em pontos percentuais e favorece cadeias grandes.

Gráfico: barras horizontais de `crescimento_relativo_perc` por `cadeia`.

`crescimento_relativo_base2021_perc` repete a medida a partir de 2021. O ano de 2020 teve arrecadação deprimida pela pandemia; cadeia que muda muito de posição entre as duas bases tem resultado dependente do ano de partida.

In [0]:
%sql
WITH total AS (
  SELECT t.ano, sum(f.valor_icms) AS icms_rs
  FROM gold.fato_icms_cnae f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  WHERE t.ano IN (2020, 2021, 2025)
  GROUP BY t.ano
),
cadeia AS (
  SELECT p.cadeia, t.ano, sum(f.valor_icms) AS icms
  FROM gold.fato_icms_cnae f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE t.ano IN (2020, 2021, 2025)
  GROUP BY p.cadeia, t.ano
),
base AS (
  SELECT c.cadeia,
         sum(CASE WHEN c.ano = 2020 THEN c.icms END)            AS icms_2020,
         sum(CASE WHEN c.ano = 2025 THEN c.icms END)            AS icms_2025,
         sum(CASE WHEN c.ano = 2020 THEN c.icms / t.icms_rs END) AS part_2020,
         sum(CASE WHEN c.ano = 2021 THEN c.icms / t.icms_rs END) AS part_2021,
         sum(CASE WHEN c.ano = 2025 THEN c.icms / t.icms_rs END) AS part_2025
  FROM cadeia c JOIN total t ON t.ano = c.ano
  GROUP BY c.cadeia
)
SELECT b.cadeia,
       round(b.icms_2020 / 1e6, 1)                                     AS icms_2020_mi,
       round(b.icms_2025 / 1e6, 1)                                     AS icms_2025_mi,
       round(100 * (b.icms_2025 / b.icms_2020 - 1), 1)                 AS var_nominal_perc,
       round(100 * (power(b.icms_2025 / b.icms_2020, 1.0 / 5) - 1), 1) AS taxa_media_anual_perc,
       round(100 * (b.part_2025 / b.part_2020 - 1), 1)                 AS crescimento_relativo_perc,
       round(100 * (b.part_2025 / b.part_2021 - 1), 1)                 AS crescimento_relativo_base2021_perc,
       rank() OVER (ORDER BY b.part_2025 / b.part_2020 DESC)           AS posicao_base2020,
       rank() OVER (ORDER BY b.part_2025 / b.part_2021 DESC)           AS posicao_base2021,
       round(100 * b.part_2020, 2)                                     AS participacao_2020_perc,
       round(100 * b.part_2025, 2)                                     AS participacao_2025_perc,
       round(100 * (b.part_2025 - b.part_2020), 2)                     AS var_participacao_pp,
       d.cobertura_icms_perc,
       d.ressalva
FROM base b
JOIN gold.dim_cadeia_produtiva d ON d.cadeia = b.cadeia
ORDER BY crescimento_relativo_perc DESC;

Moda e Alimentos e Bebidas são as únicas cadeias sem ressalva que ampliaram participação no ICMS nas duas bases de comparação. O crescimento de Metalmecânico e Casa e Construção desde 2020 reflete a recuperação da pandemia: a partir de 2021, ambas perdem participação.

### 1.3 Referência: ICMS total do RS

Variação nominal do Estado no mesmo período, base de comparação da medida relativa.

In [0]:
%sql
SELECT round(sum(CASE WHEN t.ano = 2020 THEN f.valor_icms END) / 1e9, 2) AS icms_2020_bi,
       round(sum(CASE WHEN t.ano = 2025 THEN f.valor_icms END) / 1e9, 2) AS icms_2025_bi,
       round(100 * (sum(CASE WHEN t.ano = 2025 THEN f.valor_icms END)
                  / sum(CASE WHEN t.ano = 2020 THEN f.valor_icms END) - 1), 1) AS var_nominal_perc
FROM gold.fato_icms_cnae f
JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
WHERE t.ano IN (2020, 2025);

### 1.4 Contexto: concentração do ICMS por subclasse

Participação das dez maiores subclasses no ICMS de 2024, com e sem o código residual `0000000`. Sem o residual,
o denominador é o ICMS com CNAE atribuído.

In [0]:
%sql
WITH s AS (
  SELECT f.cnae_subclasse, sum(f.valor_icms) AS v
  FROM gold.fato_icms_cnae f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  WHERE t.ano = 2024
  GROUP BY f.cnae_subclasse
),
tot AS (
  SELECT sum(v) AS total,
         sum(CASE WHEN cnae_subclasse <> '0000000' THEN v END) AS total_com_cnae
  FROM s
),
top_com AS (SELECT sum(v) AS v FROM (SELECT v FROM s ORDER BY v DESC LIMIT 10)),
top_sem AS (SELECT sum(v) AS v FROM (SELECT v FROM s WHERE cnae_subclasse <> '0000000' ORDER BY v DESC LIMIT 10))
SELECT 'Inclui o residual 0000000' AS criterio,
       round(100 * top_com.v / tot.total, 2) AS participacao_top10_perc
FROM top_com, tot
UNION ALL
SELECT 'Exclui o residual 0000000',
       round(100 * top_sem.v / tot.total_com_cnae, 2)
FROM top_sem, tot;

As dez maiores subclasses concentram 41,7% do ICMS de 2024. Excluído o código residual sem CNAE, a concentração é de 40,9% do ICMS com atividade atribuída.

## P2 — Concentração das desonerações de ICMS

Recorte: ICMS, nível detalhado, finalidade econômica e `integra_total_estadual`, 2020–2025. A fonte recomenda ler
os dados por CNAE e COREDE como tendência, não como grandeza: a análise usa participação e ordenação.

### 2.1 Tamanho do recorte por ano

`calamidade_mi` mostra o valor dos dispositivos marcados com `evento_extraordinario`.

In [0]:
%sql
WITH recorte AS (
  SELECT d.ano, d.nome_corede, d.cnae_subclasse, d.valor_desonerado, b.evento_extraordinario
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS'
    AND b.finalidade_economica
    AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
)
SELECT ano,
       round(sum(valor_desonerado) / 1e6, 1) AS recorte_mi,
       round(sum(CASE WHEN evento_extraordinario THEN valor_desonerado ELSE 0 END) / 1e6, 1) AS calamidade_mi
FROM recorte
GROUP BY ano
ORDER BY ano;

### 2.2 Por cadeia produtiva

`participacao_perc` usa como denominador o recorte inteiro, com e sem cadeia. `razao_desoneracao_icms_perc` divide
a desoneração da cadeia pelo ICMS arrecadado pela mesma cadeia no período.

Gráfico: barras de `participacao_perc` por `cadeia`.

In [0]:
%sql
WITH recorte AS (
  SELECT d.ano, d.nome_corede, d.cnae_subclasse, d.valor_desonerado, b.evento_extraordinario
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS'
    AND b.finalidade_economica
    AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
),
total AS (SELECT sum(valor_desonerado) AS total FROM recorte),
icms_cadeia AS (
  SELECT p.cadeia, sum(f.valor_icms) AS icms
  FROM gold.fato_icms_cnae f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE t.ano BETWEEN 2020 AND 2025
  GROUP BY p.cadeia
),
deson_cadeia AS (
  SELECT p.cadeia,
         sum(r.valor_desonerado) AS deson,
         sum(CASE WHEN NOT r.evento_extraordinario THEN r.valor_desonerado ELSE 0 END) AS deson_sem_calamidade
  FROM recorte r
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = r.cnae_subclasse AND p.cadeia_prioritaria
  GROUP BY p.cadeia
)
SELECT d.cadeia,
       round(d.deson / 1e6, 1)                               AS desoneracao_mi,
       round(d.deson_sem_calamidade / 1e6, 1)                AS desoneracao_sem_calamidade_mi,
       round(100 * d.deson / t.total, 2)                     AS participacao_perc,
       round(100 * d.deson / i.icms, 1)                      AS razao_desoneracao_icms_perc,
       c.cobertura_icms_perc,
       c.ressalva
FROM deson_cadeia d
CROSS JOIN total t
LEFT JOIN icms_cadeia i ON i.cadeia = d.cadeia
JOIN gold.dim_cadeia_produtiva c ON c.cadeia = d.cadeia
ORDER BY participacao_perc DESC;

Alimentos e Bebidas concentra 44,5% da desoneração setorial de ICMS. Leite e Vitivinicultura estão contidas nela, e 70% da desoneração de Pecuária está em subclasses de abate compartilhadas com ela. A desoneração setorial do RS é, na prática, a da agroindústria de alimentos e a do Metalmecânico.

### 2.3 Sobreposição entre cadeias

Desoneração em subclasses que pertencem a duas cadeias prioritárias. Esse valor aparece nas duas linhas da tabela 2.2; por isso as participações por cadeia não se somam.

In [0]:
%sql
WITH recorte AS (
  SELECT d.cnae_subclasse, d.valor_desonerado
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS' AND b.finalidade_economica AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
),
pares AS (
  SELECT a.cnae_subclasse, a.cadeia AS cadeia_1, b.cadeia AS cadeia_2
  FROM gold.ponte_cnae_cadeia a
  JOIN gold.ponte_cnae_cadeia b
    ON a.cnae_subclasse = b.cnae_subclasse AND a.cadeia < b.cadeia
  WHERE a.cadeia_prioritaria AND b.cadeia_prioritaria
)
SELECT p.cadeia_1, p.cadeia_2,
       count(DISTINCT p.cnae_subclasse)       AS subclasses,
       round(sum(r.valor_desonerado) / 1e6, 1) AS desoneracao_compartilhada_mi
FROM pares p
LEFT JOIN recorte r ON r.cnae_subclasse = p.cnae_subclasse
GROUP BY p.cadeia_1, p.cadeia_2
ORDER BY desoneracao_compartilhada_mi DESC NULLS LAST;

### 2.3 Cobertura do recorte pelas cadeias prioritárias

Parcela da desoneração do recorte em subclasses de ao menos uma cadeia prioritária, sem dupla contagem.

In [0]:
%sql
WITH recorte AS (
  SELECT d.ano, d.nome_corede, d.cnae_subclasse, d.valor_desonerado, b.evento_extraordinario
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS'
    AND b.finalidade_economica
    AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
),
subclasses AS (SELECT DISTINCT cnae_subclasse FROM gold.ponte_cnae_cadeia WHERE cadeia_prioritaria)
SELECT round(sum(r.valor_desonerado) / 1e6, 1)                                                 AS recorte_mi,
       round(sum(CASE WHEN s.cnae_subclasse IS NOT NULL THEN r.valor_desonerado END) / 1e6, 1) AS em_cadeia_prioritaria_mi,
       round(100 * sum(CASE WHEN s.cnae_subclasse IS NOT NULL THEN r.valor_desonerado END)
                 / sum(r.valor_desonerado), 1)                                                 AS cobertura_perc
FROM recorte r
LEFT JOIN subclasses s ON s.cnae_subclasse = r.cnae_subclasse;

### 2.3 Conferência dos nomes de COREDE

A desoneração publica o COREDE por nome, sem acento; a dimensão de município traz o nome do cadastro. A junção usa
uma chave sem acento, sem preposições e sem sinais. Resultado esperado: nenhuma linha.

In [0]:
%sql
SELECT DISTINCT d.nome_corede
FROM gold.fato_desoneracao_detalhada d
WHERE d.nome_corede IS NOT NULL
  AND regexp_replace(upper(translate(trim(d.nome_corede), 'ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç','AAAAEEIOOOUCaaaaeeiooouc')), '\\b(DA|DE|DO|DAS|DOS)\\b|[^A-Z]', '') NOT IN (
    SELECT regexp_replace(upper(translate(trim(m.nome_corede), 'ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç','AAAAEEIOOOUCaaaaeeiooouc')), '\\b(DA|DE|DO|DAS|DOS)\\b|[^A-Z]', '') FROM gold.dim_municipio m WHERE m.nome_corede IS NOT NULL);

### 2.6 Por COREDE

A arrecadação de ICMS do COREDE vem de `fato_arrecadacao_municipio`, agregada pela dimensão de município.
`participacao_acumulada_perc` mostra quantos COREDEs concentram a maior parte do recorte.

Gráfico: barras de `participacao_perc` por `nome_corede`.

In [0]:
%sql
WITH recorte AS (
  SELECT d.ano, d.nome_corede, d.cnae_subclasse, d.valor_desonerado, b.evento_extraordinario
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS'
    AND b.finalidade_economica
    AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
),
total AS (SELECT sum(valor_desonerado) AS total FROM recorte),
deson_corede AS (
  SELECT regexp_replace(upper(translate(trim(nome_corede), 'ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç','AAAAEEIOOOUCaaaaeeiooouc')), '\\b(DA|DE|DO|DAS|DOS)\\b|[^A-Z]', '') AS chave, max(nome_corede) AS nome_corede, sum(valor_desonerado) AS deson
  FROM recorte
  GROUP BY 1
),
icms_corede AS (
  SELECT regexp_replace(upper(translate(trim(m.nome_corede), 'ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç','AAAAEEIOOOUCaaaaeeiooouc')), '\\b(DA|DE|DO|DAS|DOS)\\b|[^A-Z]', '') AS chave, sum(a.valor_arrecadado) AS icms
  FROM gold.fato_arrecadacao_municipio a
  JOIN gold.dim_tempo t ON t.sk_tempo = a.sk_tempo
  JOIN gold.dim_municipio m ON m.cod_municipio_ibge = a.cod_municipio_ibge
  WHERE upper(a.tributo) = 'ICMS' AND t.ano BETWEEN 2020 AND 2025
  GROUP BY 1
)
SELECT d.nome_corede,
       round(d.deson / 1e6, 1)                         AS desoneracao_mi,
       round(100 * d.deson / t.total, 2)               AS participacao_perc,
       round(100 * sum(d.deson) OVER (ORDER BY d.deson DESC) / t.total, 1) AS participacao_acumulada_perc,
       round(100 * d.deson / i.icms, 1)                AS razao_desoneracao_icms_perc
FROM deson_corede d
CROSS JOIN total t
LEFT JOIN icms_corede i ON i.chave = d.chave
ORDER BY d.deson DESC;

Em valor, a desoneração setorial acompanha o tamanho da economia regional: cinco COREDEs concentram 60%. Em intensidade, o padrão se inverte: nos COREDEs agroindustriais do norte, o valor renunciado chega a 55%–76% do ICMS arrecadado, contra 5%–10% na Região Metropolitana e no Vale do Sinos.

## P3 — Participação de MPE nas empresas ativas

Universo: empresas de categoria Simples Nacional e Geral (`entra_analise` e não produtor rural). MPE: Simples
Nacional. Estoque lido no `snapshot_ano` de 2020 e de 2025.

Em 01/01/2024, a Receita Federal excluiu do Simples Nacional as empresas com débitos não regularizados. No RS, cerca de 14,8 mil estabelecimentos passaram do Simples Nacional para a Geral sem registro de abertura ou baixa. A participação de MPE no Estado é estável de 2020 a 2023 (76,7% e 76,2%) e cai para 72,1% em 2025: a queda entre 2020 e 2025 reflete mudança de regime tributário, não de porte.



### 3.1 Por cadeia produtiva

In [0]:
%sql
WITH estoque AS (
  SELECT p.cadeia, t.ano,
         sum(CASE WHEN c.e_mpe THEN f.qtd_ativos ELSE 0 END) AS mpe,
         sum(f.qtd_ativos) AS total
  FROM gold.fato_cadastro_setor f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE f.snapshot_ano AND t.ano IN (2020, 2025)
    AND c.entra_analise AND NOT c.e_produtor_rural
  GROUP BY p.cadeia, t.ano
)
SELECT cadeia,
       sum(CASE WHEN ano = 2025 THEN total END)                           AS empresas_2025,
       round(100 * sum(CASE WHEN ano = 2020 THEN mpe END)
                 / sum(CASE WHEN ano = 2020 THEN total END), 1)           AS participacao_mpe_2020_perc,
       round(100 * sum(CASE WHEN ano = 2025 THEN mpe END)
                 / sum(CASE WHEN ano = 2025 THEN total END), 1)           AS participacao_mpe_2025_perc
FROM estoque
GROUP BY cadeia
ORDER BY participacao_mpe_2025_perc DESC;

### 3.2 Por COREDE

Gráfico sugerido: barras de `participacao_mpe_2025_perc` por `nome_corede`.

In [0]:
%sql
WITH estoque AS (
  SELECT m.nome_corede, t.ano,
         sum(CASE WHEN c.e_mpe THEN f.qtd_ativos ELSE 0 END) AS mpe,
         sum(f.qtd_ativos) AS total
  FROM gold.fato_cadastro_municipio f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
  JOIN gold.dim_municipio m ON m.cod_municipio_ibge = f.cod_municipio_ibge
  WHERE f.snapshot_ano AND t.ano IN (2020, 2025)
    AND c.entra_analise AND NOT c.e_produtor_rural
    AND m.nome_corede IS NOT NULL
  GROUP BY m.nome_corede, t.ano
)
SELECT nome_corede,
       sum(CASE WHEN ano = 2025 THEN total END)                           AS empresas_2025,
       round(100 * sum(CASE WHEN ano = 2020 THEN mpe END)
                 / sum(CASE WHEN ano = 2020 THEN total END), 1)           AS participacao_mpe_2020_perc,
       round(100 * sum(CASE WHEN ano = 2025 THEN mpe END)
                 / sum(CASE WHEN ano = 2025 THEN total END), 1)           AS participacao_mpe_2025_perc
FROM estoque
GROUP BY nome_corede
ORDER BY participacao_mpe_2025_perc DESC;

### 3.3 Evolução da abertura de MPE no RS

Fluxo anual do Simples Nacional, pelo cadastro por setor. O cadastro por município não é usado aqui: seus fluxos coincidem com os do cadastro por setor em 2020, ficam 20% abaixo em 2021 e caem para menos de 10% a partir de 2022, enquanto o estoque das duas tabelas segue próximo.

Aberturas e baixas incluem migração entre categorias, declarada pela fonte. As baixas de 2024 incluem cerca de 12,7 mil baixas de ofício em março e abril. A exclusão por débitos de jan/2024 não aparece no fluxo.

Gráfico: barras de `aberturas` e `baixas`, linha de `saldo`, por `ano`.

In [0]:
%sql
SELECT t.ano,
       sum(f.qtd_novos)                       AS aberturas,
       sum(f.qtd_baixados)                    AS baixas,
       sum(f.qtd_novos) - sum(f.qtd_baixados) AS saldo
FROM gold.fato_cadastro_setor f
JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
WHERE c.e_mpe AND c.entra_analise
  AND t.ano BETWEEN 2020 AND 2025
GROUP BY t.ano
ORDER BY t.ano;

A abertura de MPE no RS é estável, em torno de 27 mil por ano desde 2021. A redução do estoque em 2024 vem das saídas: exclusão do Simples Nacional por débitos em janeiro, cerca de 14,8 mil empresas, e baixas de ofício em março e abril, cerca de 12,7 mil, somadas ao ano da calamidade.

## P4 — Desoneração e saldo de empresas por cadeia

Intensidade de desoneração: razão entre a desoneração do recorte e o ICMS da cadeia, 2020–2025. Dinâmica
empresarial: variação do estoque de empresas (Simples Nacional e Geral) entre dez/2020 e dez/2025 e saldo
acumulado de aberturas menos baixas de 2021 a 2025, sobre o estoque de 2020.

Limitação: a desoneração setorial vem da GIA e alcança só empresas de categoria Geral; o saldo inclui as MPE.

### 4.1 Tabela por cadeia

In [0]:
%sql
WITH recorte AS (
  SELECT d.ano, d.nome_corede, d.cnae_subclasse, d.valor_desonerado, b.evento_extraordinario
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS'
    AND b.finalidade_economica
    AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
),
icms_cadeia AS (
  SELECT p.cadeia, sum(f.valor_icms) AS icms
  FROM gold.fato_icms_cnae f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE t.ano BETWEEN 2020 AND 2025
  GROUP BY p.cadeia
),
deson_cadeia AS (
  SELECT p.cadeia, sum(r.valor_desonerado) AS deson
  FROM recorte r
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = r.cnae_subclasse AND p.cadeia_prioritaria
  GROUP BY p.cadeia
),
cadastro AS (
  SELECT p.cadeia,
         sum(CASE WHEN f.snapshot_ano AND t.ano = 2020 THEN f.qtd_ativos END)      AS estoque_2020,
         sum(CASE WHEN f.snapshot_ano AND t.ano = 2025 THEN f.qtd_ativos END)      AS estoque_2025,
         sum(CASE WHEN t.ano BETWEEN 2021 AND 2025 THEN f.qtd_novos - f.qtd_baixados END) AS saldo
  FROM gold.fato_cadastro_setor f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE c.entra_analise AND NOT c.e_produtor_rural
    AND t.ano BETWEEN 2020 AND 2025
  GROUP BY p.cadeia
)
SELECT k.cadeia,
       round(100 * d.deson / i.icms, 1)                          AS intensidade_desoneracao_perc,
       k.estoque_2020,
       k.estoque_2025,
       round(100 * (k.estoque_2025 / k.estoque_2020 - 1), 1)     AS var_estoque_perc,
       round(100 * k.saldo / k.estoque_2020, 1)                  AS saldo_sobre_estoque_perc,
       c.ressalva
FROM cadastro k
LEFT JOIN deson_cadeia d ON d.cadeia = k.cadeia
LEFT JOIN icms_cadeia i  ON i.cadeia = k.cadeia
JOIN gold.dim_cadeia_produtiva c ON c.cadeia = k.cadeia
ORDER BY intensidade_desoneracao_perc DESC;

### 4.2 Correlação de Spearman

Correlação entre as posições das cadeias em intensidade de desoneração e em dinâmica empresarial. Valores próximos
de +1 indicam que as cadeias mais desoneradas também têm maior crescimento de empresas. Com 14 cadeias, só
|ρ| acima de cerca de 0,54 é significativo a 5%.

Gráfico: dispersão da tabela 4.1, `intensidade_desoneracao_perc` × `var_estoque_perc`.

In [0]:
%sql
WITH recorte AS (
  SELECT d.ano, d.nome_corede, d.cnae_subclasse, d.valor_desonerado, b.evento_extraordinario
  FROM gold.fato_desoneracao_detalhada d
  JOIN gold.dim_beneficio b
    ON b.imposto = d.imposto AND b.tipo_beneficio = d.tipo_beneficio AND b.cod_beneficio = d.cod_beneficio
  WHERE d.imposto = 'ICMS'
    AND b.finalidade_economica
    AND b.integra_total_estadual
    AND d.ano BETWEEN 2020 AND 2025
),
icms_cadeia AS (
  SELECT p.cadeia, sum(f.valor_icms) AS icms
  FROM gold.fato_icms_cnae f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE t.ano BETWEEN 2020 AND 2025
  GROUP BY p.cadeia
),
deson_cadeia AS (
  SELECT p.cadeia, sum(r.valor_desonerado) AS deson
  FROM recorte r
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = r.cnae_subclasse AND p.cadeia_prioritaria
  GROUP BY p.cadeia
),
cadastro AS (
  SELECT p.cadeia,
         sum(CASE WHEN f.snapshot_ano AND t.ano = 2020 THEN f.qtd_ativos END)      AS estoque_2020,
         sum(CASE WHEN f.snapshot_ano AND t.ano = 2025 THEN f.qtd_ativos END)      AS estoque_2025,
         sum(CASE WHEN t.ano BETWEEN 2021 AND 2025 THEN f.qtd_novos - f.qtd_baixados END) AS saldo
  FROM gold.fato_cadastro_setor f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
  JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse AND p.cadeia_prioritaria
  WHERE c.entra_analise AND NOT c.e_produtor_rural
    AND t.ano BETWEEN 2020 AND 2025
  GROUP BY p.cadeia
),
base AS (
  SELECT coalesce(d.deson, 0) / i.icms           AS intensidade,
         k.estoque_2025 / k.estoque_2020 - 1     AS var_estoque,
         k.saldo / k.estoque_2020                AS saldo_rel
  FROM cadastro k
  LEFT JOIN deson_cadeia d ON d.cadeia = k.cadeia
  JOIN icms_cadeia i ON i.cadeia = k.cadeia
),
postos AS (
  SELECT rank() OVER (ORDER BY intensidade) AS r_int,
         rank() OVER (ORDER BY var_estoque) AS r_est,
         rank() OVER (ORDER BY saldo_rel)   AS r_sal
  FROM base
)
SELECT count(*)                           AS cadeias,
       round(corr(r_int, r_est), 2)       AS spearman_intensidade_var_estoque,
       round(corr(r_int, r_sal), 2)       AS spearman_intensidade_saldo
FROM postos;

Não há associação entre intensidade de desoneração e crescimento do número de empresas por cadeia (ρ de Spearman ≈ −0,2, sem significância). As cadeias com maior renúncia relativa, Leite e Pecuária, reduziram o número de estabelecimentos, e a de maior crescimento, Metalmecânico, tem intensidade moderada. O instrumento dominante, o crédito presumido, alcança só empresas da categoria Geral e não atinge diretamente as MPE, que formam a maior parte do estoque.

## P5 — PIB per capita e abertura de pequenos negócios por município

Crescimento do PIB per capita de 2018 a 2021, dentro da mesma base populacional (estimativa do IBGE). Dinâmica de pequenos negócios: variação do estoque do Simples Nacional entre dez/2017 e dez/2021.

O estoque substitui a taxa de abertura porque os fluxos do cadastro por município são incompletos a partir de 2021 (seção 3.3). O período 2022–2023 fica fora: a população muda de base com o Censo 2022 e o estoque de dez/2023 já incorpora a exclusão do Simples Nacional por débitos.

### 5.1 Correlação de Spearman nos dois períodos

In [0]:
%sql
WITH pib AS (
  SELECT cod_municipio_ibge,
         max(CASE WHEN ano = 2018 THEN pib_per_capita_reais END) AS ppc_2018,
         max(CASE WHEN ano = 2021 THEN pib_per_capita_reais END) AS ppc_2021
  FROM gold.fato_pib_municipal
  GROUP BY cod_municipio_ibge
),
mpe AS (
  SELECT f.cod_municipio_ibge,
         sum(CASE WHEN t.ano = 2017 THEN f.qtd_ativos END) AS estoque_2017,
         sum(CASE WHEN t.ano = 2021 THEN f.qtd_ativos END) AS estoque_2021
  FROM gold.fato_cadastro_municipio f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
  WHERE f.snapshot_ano AND t.ano IN (2017, 2021)
    AND c.e_mpe AND c.entra_analise AND f.cod_municipio_ibge IS NOT NULL
  GROUP BY f.cod_municipio_ibge
),
base AS (
  SELECT p.ppc_2021 / p.ppc_2018 - 1          AS cresc_ppc,
         m.estoque_2021 / m.estoque_2017 - 1  AS var_estoque
  FROM pib p JOIN mpe m ON m.cod_municipio_ibge = p.cod_municipio_ibge
  WHERE m.estoque_2017 > 0 AND p.ppc_2018 > 0
),
postos AS (
  SELECT rank() OVER (ORDER BY cresc_ppc)   AS rx,
         rank() OVER (ORDER BY var_estoque) AS ry
  FROM base
)
SELECT count(*)                AS municipios,
       round(corr(rx, ry), 2)  AS spearman
FROM postos;

### 5.2 Quintis de crescimento do PIB per capita

Municípios divididos em cinco grupos pelo crescimento do PIB per capita; mediana da variação do estoque em cada grupo. Se a mediana sobe do quintil 1 ao 5, a associação é positiva.

Gráfico: barras de `mediana_var_estoque_perc` por `quintil`.

In [0]:
%sql
WITH pib AS (
  SELECT cod_municipio_ibge,
         max(CASE WHEN ano = 2018 THEN pib_per_capita_reais END) AS ppc_2018,
         max(CASE WHEN ano = 2021 THEN pib_per_capita_reais END) AS ppc_2021
  FROM gold.fato_pib_municipal
  GROUP BY cod_municipio_ibge
),
mpe AS (
  SELECT f.cod_municipio_ibge,
         sum(CASE WHEN t.ano = 2017 THEN f.qtd_ativos END) AS estoque_2017,
         sum(CASE WHEN t.ano = 2021 THEN f.qtd_ativos END) AS estoque_2021
  FROM gold.fato_cadastro_municipio f
  JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
  JOIN gold.dim_categoria_contribuinte c ON c.categoria = f.categoria
  WHERE f.snapshot_ano AND t.ano IN (2017, 2021)
    AND c.e_mpe AND c.entra_analise AND f.cod_municipio_ibge IS NOT NULL
  GROUP BY f.cod_municipio_ibge
),
base AS (
  SELECT p.ppc_2021 / p.ppc_2018 - 1          AS cresc_ppc,
         m.estoque_2021 / m.estoque_2017 - 1  AS var_estoque
  FROM pib p JOIN mpe m ON m.cod_municipio_ibge = p.cod_municipio_ibge
  WHERE m.estoque_2017 > 0 AND p.ppc_2018 > 0
),
q AS (SELECT *, ntile(5) OVER (ORDER BY cresc_ppc) AS quintil FROM base)
SELECT quintil,
       count(*)                                            AS municipios,
       round(100 * percentile_approx(cresc_ppc, 0.5), 1)   AS mediana_cresc_ppc_perc,
       round(100 * percentile_approx(var_estoque, 0.5), 1) AS mediana_var_estoque_perc
FROM q
GROUP BY quintil
ORDER BY quintil;

Não há associação entre o crescimento do PIB per capita e a variação do número de empresas do Simples Nacional nos municípios entre 2018 e 2021. A correlação de Spearman é 0,02, em 497 municípios. Nos cinco grupos de municípios ordenados pelo crescimento do PIB per capita, a variação mediana do estoque fica entre −1,8% e 0%.
